# **1. SciREX**


## 1.1 Загрузка датасета

In [ ]:
# Шаг 1: Скачиваем и распаковываем датасет
!wget https://github.com/allenai/SciREX/raw/master/scirex_dataset/release_data.tar.gz -O scirex_dataset.tar.gz
!mkdir scirex_dataset && tar -xvzf scirex_dataset.tar.gz -C scirex_dataset

--2025-06-06 17:36:19--  https://github.com/allenai/SciREX/raw/master/scirex_dataset/release_data.tar.gz
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/allenai/SciREX/master/scirex_dataset/release_data.tar.gz [following]
--2025-06-06 17:36:20--  https://raw.githubusercontent.com/allenai/SciREX/master/scirex_dataset/release_data.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7318844 (7.0M) [application/octet-stream]
Saving to: ‘scirex_dataset.tar.gz’

scirex_dataset.tar. 100%[===================>]   6.98M  --.-KB/s    in 0.09s   

2025-06-06 17:36:20 (76.7 MB/s) - ‘scirex_dataset.tar.gz’

In [ ]:
# Шаг 2: Загружаем JSONL-файлы
import json

def load_scirex(split: str, path: str = "scirex_dataset/release_data/"):
    data = []
    with open(f"{path}/{split}.jsonl", 'r') as f:
        for line in f:
            doc = json.loads(line)
            data.append(doc)
    return data

In [ ]:
train_docs = load_scirex("train")
print(f"Загружено документов: {len(train_docs)}")

Загружено документов: 306


------

In [ ]:
import re
import pandas as pd

In [ ]:
def detokenize_words(words):
    punctuation = {'.', ',', ':', ';', '!', '?', '%', ')', ']', '}', '»', '...'}
    opening_punct = {'(', '[', '{', '«', '“', '"', "'"}

    text = ""
    for i, token in enumerate(words):
        token = str(token)
        if i == 0:
            text += token
        elif token in punctuation:
            text += token
        elif words[i-1] in opening_punct:
            text += token
        else:
            text += " " + token
    text = re.sub(r'\s+([.,!?:;])', r'\1', text)
    return text

In [ ]:
def extract_clean_row(doc):
    row = {
        "doc_id": doc.get("doc_id", ""),
        "methods": [],
        "tasks": [],
        "metrics": [],
        "materials": [],
        "score": [],
    }

    # Маппинг ключей из n_ary_relations в нужные поля
    key_map = {
        "Method": "methods",
        "Task": "tasks",
        "Metric": "metrics",
        "Material": "materials",
        "score": "score",  # если хочешь объединять материалы и датасеты
    }

    for rel in doc.get("n_ary_relations", []):
        for key, value in rel.items():
            if key in key_map and value:
                target = key_map[key]
                if isinstance(value, list):
                    row[target].extend(v.strip() for v in value if isinstance(v, str))
                elif isinstance(value, str):
                    row[target].append(value.strip())

    # Удалим дубли
    for k in ["methods", "tasks", "metrics", "materials", "score"]:
        row[k] = list(set(row[k]))

    # Восстановим оригинальный текст
    words = doc.get("words", [])
    row["original_text"] = detokenize_words(words) if words else ""

    # Добавим прочие поля
    extra_fields = ['coref', 'coref_non_salient', 'method_subrelations',
                    'n_ary_relations', 'ner', 'sections', 'sentences', 'words']

    for field in extra_fields:
        row[field] = doc.get(field)

    return row


In [ ]:
rows = [extract_clean_row(doc) for doc in train_docs]
df = pd.DataFrame(rows)

In [ ]:
print("Форма таблицы:", df.shape)
print("Столбцы:", df.columns.tolist())
df[["doc_id", "methods", "tasks", "metrics", "materials", "score", "original_text"]].head(3)

Форма таблицы: (306, 15)
Столбцы: ['doc_id', 'methods', 'tasks', 'metrics', 'materials', 'score', 'original_text', 'coref', 'coref_non_salient', 'method_subrelations', 'n_ary_relations', 'ner', 'sections', 'sentences', 'words']


,doc_id,methods,tasks,metrics,materials,score,original_text
0,000f90380d768a85e2316225854fc377c079b5c4,[FRRN],"[Semantic_Segmentation, Real-Time_Semantic_Seg...","[Frame__fps_, Mean_IoU, mIoU, Time__ms_]",[Cityscapes],"[2.1, 71.8%, 469]",Full - Resolution Residual Networks for Semant...
1,0012de6bec1f25599e4f02517637e531a71909b9,[V-Net___Dice-based_loss],[Volumetric_Medical_Image_Segmentation],[Dice_Score],[PROMISE_2012],[0.869],document: V - Net: Fully Convolutional Neural ...
2,007ab5528b3bd310a80d553cccad4b78dc496b02,"[BiDAF__single_model_, BiDAF, BiDAF__ensemble_...","[Question_Answering, Open-Domain_Question_Answ...","[METEOR, EM__Quasar-T_, F1__Quasar-T_, F1, EM,...","[NarrativeQA, MS_MARCO, SQuAD1_1, CNN___Daily_...","[33.45, 23.96, 15.69, 25.9, 79.6, 36.74, 76.9,...",document: Bi - Directional Attention Flow for ...


In [ ]:
df["original_text"][0]

"Full - Resolution Residual Networks for Semantic Segmentation in Street Scenes section: Abstract Semantic image segmentation is an essential component of modern autonomous driving systems, as an accurate understanding of the surrounding scene is crucial to navigation and action planning. Current state - of - the - art approaches in semantic image segmentation rely on pretrained networks that were initially developed for classifying images as a whole. While these networks exhibit outstanding recognition performance (i.e., what is visible?), they lack localization accuracy (i.e., where precisely is something located?). Therefore, additional processing steps have to be performed in order to obtain pixel - accurate segmentation masks at the full image resolution. To alleviate this problem we propose a novel ResNet - like architecture that exhibits strong localization and recognition performance. We combine multi - scale context with pixel - level accuracy by using two processing streams w

In [ ]:
import re
from collections import Counter

def compare_tokens_from_df(df_row, doc):
    """
    Сравнивает токены из doc["words"] и токены из df["original_text"]
    """
    tokens_original = list(map(str, doc.get("words", [])))

    # Получаем текст из DataFrame
    text_from_df = df_row["original_text"]

    # Токенизируем текст из DataFrame
    tokens_reconstructed = re.findall(r'\w+|[^\w\s]', text_from_df, re.UNICODE)

    # Подсчёт токенов
    counter_original = Counter(tokens_original)
    counter_reconstructed = Counter(tokens_reconstructed)

    missing = counter_original - counter_reconstructed
    extra = counter_reconstructed - counter_original

    is_equal = not missing and not extra

    print("=" * 50)
    print(f"doc_id: {df_row.get('doc_id', '')}")
    print(f"Токенов в оригинале: {len(tokens_original)}")
    print(f"Токенов в восстановленном тексте: {len(tokens_reconstructed)}")
    print("✅ Совпадают" if is_equal else "❌ НЕ совпадают")

    if missing:
        print("\n🔺 Отсутствующие токены:")
        for token, count in missing.items():
            print(f"  {token!r} — {count} шт.")
    if extra:
        print("\n🔻 Лишние токены:")
        for token, count in extra.items():
            print(f"  {token!r} — {count} шт.")
    print("=" * 50)

    return is_equal

In [ ]:
for i in range(5):
    df_row = df.iloc[i]
    doc = train_docs[i]
    compare_tokens_from_df(df_row, doc)

doc_id: 000f90380d768a85e2316225854fc377c079b5c4
Токенов в оригинале: 6507
Токенов в восстановленном тексте: 6634
❌ НЕ совпадают

🔺 Отсутствующие токены:
  'i.e.' — 3 шт.
  '71.8' — 1 шт.
  '][' — 26 шт.
  'e.g.' — 2 шт.
  'https:' — 1 шт.
  '//' — 1 шт.
  'github.com' — 1 шт.
  'n−1' — 5 шт.
  'F.' — 1 шт.
  "'s" — 6 шт.
  'G.' — 1 шт.
  '∂H' — 1 шт.
  'zi;Wi' — 1 шт.
  '∂zn' — 1 шт.
  '∂l' — 1 шт.
  '∂zm' — 1 шт.
  'H.' — 3 шт.
  '...' — 4 шт.
  'i.' — 1 шт.
  '|{i' — 1 шт.
  'K.' — 2 шт.
  'A.' — 2 шт.
  '16.7' — 1 шт.
  'vs.' — 1 шт.
  '17.7' — 1 шт.
  '−3' — 1 шт.
  '−4' — 1 шт.
  '65.7' — 1 шт.
  '62.8' — 1 шт.
  'B.' — 2 шт.
  'r.' — 1 шт.
  '0.5' — 7 шт.
  '−0.5' — 2 шт.
  '−a' — 1 шт.
  '5.2' — 1 шт.
  'C.' — 1 шт.

🔻 Лишние токены:
  ':' — 1 шт.
  '.' — 53 шт.
  'a' — 1 шт.
  'i' — 5 шт.
  'e' — 5 шт.
  '71' — 1 шт.
  '8' — 2 шт.
  'A' — 2 шт.
  '[' — 26 шт.
  ']' — 26 шт.
  'g' — 2 шт.
  ';' — 1 шт.
  '1' — 5 шт.
  '/' — 2 шт.
  'https' — 1 шт.
  'github' — 1 шт.
  'com' — 1

------

##1.2 Сохранение датасета

In [ ]:
df_to_save = df[[
    "doc_id", "methods", "tasks", "metrics", "materials", "score",
    "original_text"
]]

# Сохраняем как JSON (списки и строки сохранятся как есть)
df_to_save.to_json("scirex_structured.json", orient="records", lines=True, force_ascii=False)

##1.3 Загрузка датасета

In [ ]:
import pandas as pd

df_loaded = pd.read_json("scirex_structured.json", orient="records", lines=True)

In [ ]:
df_loaded.head()

,doc_id,methods,tasks,metrics,materials,score,original_text
0,000f90380d768a85e2316225854fc377c079b5c4,[FRRN],"[Semantic_Segmentation, Real-Time_Semantic_Seg...","[Frame__fps_, Mean_IoU, mIoU, Time__ms_]",[Cityscapes],"[2.1, 71.8%, 469]",Full - Resolution Residual Networks for Semant...
1,0012de6bec1f25599e4f02517637e531a71909b9,[V-Net___Dice-based_loss],[Volumetric_Medical_Image_Segmentation],[Dice_Score],[PROMISE_2012],[0.869],document: V - Net: Fully Convolutional Neural ...
2,007ab5528b3bd310a80d553cccad4b78dc496b02,"[BiDAF__single_model_, BiDAF, BiDAF__ensemble_...","[Question_Answering, Open-Domain_Question_Answ...","[METEOR, EM__Quasar-T_, F1__Quasar-T_, F1, EM,...","[NarrativeQA, MS_MARCO, SQuAD1_1, CNN___Daily_...","[33.45, 23.96, 15.69, 25.9, 79.6, 36.74, 76.9,...",document: Bi - Directional Attention Flow for ...
3,0095c269e7d0c990249312687fc43521019809c4,[50D_stacked_TC-LSTMs],[Natural_Language_Inference],"[Parameters, __Train_Accuracy, __Test_Accuracy]",[SNLI],"[190k, 86.7, 85.1]",document: Modelling Interaction of Sentence Pa...
4,00b1cdc5bd77bf27f9b1ca630365eeeb456913b4,"[AlphaZero, AlphaGo_Zero]","[Game_of_Go, Game_of_Shogi]",[ELO_Rating],[ELO_Ratings],"[4650, 5185]",document: Mastering Chess and Shogi by Self - ...


In [ ]:
len(df_loaded)

306

-------------------------------